In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q tokenizers datasets accelerate

## cell 2

In [3]:
import torch, math, os, time, json

CONFIG = dict(
    vocab_size = 32000,
    block_size = 512,
    n_layer = 12,
    n_head  = 12,
    n_embd  = 768,
    dropout = 0.1,
    bias    = True,
    use_rope = False,      # <-- FALSE = baseline run, TRUE = RoPE run. Run notebook once each.
)

BATCH_SIZE       = 16
GRAD_ACCUM_STEPS = 4       # effective batch = 64
MAX_STEPS        = 8000   # set this AFTER the timing test in Cell 7
WARMUP_STEPS     = 300
LR               = 3e-4

LANGS   = ["en", "hi", "mr"]                    # English, Hindi, Marathi
RUN_TAG = "rope" if CONFIG["use_rope"] else "baseline"
CKPT_DIR = f"/kaggle/working/ckpt_{RUN_TAG}"
TOK_PATH = "/kaggle/working/tokenizer"
os.makedirs(CKPT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.cuda.get_device_name(0) if device=="cuda" else "")

cuda Tesla T4


## Cell 3

In [4]:
from datasets import load_dataset

# Unicode ranges we care about
def _frac_in_range(text, lo, hi):
    letters = [c for c in text if c.isalpha()]
    if not letters:
        return 0.0
    inrange = sum(lo <= ord(c) <= hi for c in letters)
    return inrange / len(letters)

def is_lang(text, lang_code, min_frac=0.6):
    t = text.replace("\n", " ")[:500]
    if not t.strip():
        return False
    if lang_code == "en":
        # mostly basic Latin letters
        return _frac_in_range(t, 0x0041, 0x007A) >= min_frac
    elif lang_code in ("hi", "mr"):
        # Devanagari block U+0900–U+097F (hi & mr share it; the c4 split separates them)
        return _frac_in_range(t, 0x0900, 0x097F) >= min_frac
    return True   # fallback: accept

def lang_stream(lang):
    ds = load_dataset("allenai/c4", lang, split="train", streaming=True)
    for ex in ds:
        if is_lang(ex["text"], lang):
            yield ex["text"]

## Cell 4

In [5]:
from tokenizers import ByteLevelBPETokenizer
import itertools

if not os.path.exists(f"{TOK_PATH}/vocab.json"):
    os.makedirs(TOK_PATH, exist_ok=True)
    print("collecting tokenizer training sample...")
    with open("/kaggle/working/tok_train.txt", "w", encoding="utf-8") as f:
        for lang in LANGS:
            n = 0
            for text in lang_stream(lang):     # already language-filtered
                f.write(text.replace("\n", " ") + "\n")
                n += 1
                if n >= 20000:                 # ~20k clean docs per language
                    break
            print(f"  {lang}: {n} docs")

    tok = ByteLevelBPETokenizer()
    tok.train(files=["/kaggle/working/tok_train.txt"],
              vocab_size=CONFIG["vocab_size"], min_frequency=2,
              special_tokens=["<|endoftext|>"])
    tok.save_model(TOK_PATH)
    print("tokenizer saved")
else:
    print("tokenizer already exists, skipping")

collecting tokenizer training sample...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

  en: 20000 docs


...

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

  hi: 20000 docs


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

  mr: 20000 docs



tokenizer saved


## Cell 5

In [6]:
from torch.utils.data import IterableDataset, DataLoader

tok = ByteLevelBPETokenizer(f"{TOK_PATH}/vocab.json", f"{TOK_PATH}/merges.txt")
EOT_ID = tok.token_to_id("<|endoftext|>")

def interleaved_stream():
    streams = [lang_stream(l) for l in LANGS]   # round-robin: each batch sees a mix
    for texts in itertools.zip_longest(*streams):
        for t in texts:
            if t:
                yield t

class PackedTokenDataset(IterableDataset):
    def __init__(self, block_size):
        self.block_size = block_size
    def __iter__(self):
        buf = []
        for text in interleaved_stream():
            buf.extend(tok.encode(text).ids + [EOT_ID])
            while len(buf) >= self.block_size + 1:
                chunk = buf[:self.block_size + 1]
                buf = buf[self.block_size:]
                yield (torch.tensor(chunk[:-1], dtype=torch.long),
                       torch.tensor(chunk[1:],  dtype=torch.long))

train_loader = DataLoader(PackedTokenDataset(CONFIG["block_size"]),
                          batch_size=BATCH_SIZE, num_workers=2)

## Cell 6

In [7]:
import torch.nn as nn
import torch.nn.functional as F

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(q, k, cos, sin):
    return (q*cos) + (rotate_half(q)*sin), (k*cos) + (rotate_half(k)*sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :])
        self.register_buffer("sin", emb.sin()[None, None, :, :])
    def forward(self, seq_len):
        return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head, self.n_embd = cfg["n_head"], cfg["n_embd"]
        self.head_dim = self.n_embd // self.n_head
        self.qkv  = nn.Linear(self.n_embd, 3*self.n_embd, bias=cfg["bias"])
        self.proj = nn.Linear(self.n_embd, self.n_embd, bias=cfg["bias"])
        self.attn_drop  = nn.Dropout(cfg["dropout"])
        self.resid_drop = nn.Dropout(cfg["dropout"])
        self.use_rope = cfg["use_rope"]
        if self.use_rope:
            self.rope = RotaryEmbedding(self.head_dim, cfg["block_size"])
    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if self.use_rope:
            cos, sin = self.rope(T)
            q, k = apply_rope(q, k, cos, sin)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.attn_drop.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc   = nn.Linear(cfg["n_embd"], 4*cfg["n_embd"], bias=cfg["bias"])
        self.proj = nn.Linear(4*cfg["n_embd"], cfg["n_embd"], bias=cfg["bias"])
        self.drop = nn.Dropout(cfg["dropout"])
    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg["n_embd"])
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg["n_embd"])
        self.mlp  = MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.use_rope = cfg["use_rope"]
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["n_embd"])
        if not self.use_rope:
            self.pos_emb = nn.Embedding(cfg["block_size"], cfg["n_embd"])
        self.drop   = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg["n_layer"])])
        self.ln_f   = nn.LayerNorm(cfg["n_embd"])
        self.head   = nn.Linear(cfg["n_embd"], cfg["vocab_size"], bias=False)
        self.tok_emb.weight = self.head.weight    # weight tying
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if not self.use_rope:
            x = x + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

model = GPT(CONFIG).to(device)
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M  (use_rope={CONFIG['use_rope']})")

Params: 110.03M  (use_rope=False)


## Cell 7

In [8]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.1)
scaler = torch.cuda.amp.GradScaler()   # required for fp16
data_iter = iter(train_loader)

model.train()
torch.cuda.synchronize(); t0 = time.time()
for _ in range(50):
    x, y = next(data_iter); x, y = x.to(device), y.to(device)
    with torch.autocast(device_type="cuda", dtype=torch.float16):   # fp16, NOT bf16 (P100)
        _, loss = model(x, y)
    scaler.scale(loss).backward()
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
torch.cuda.synchronize()
sec_per_step = (time.time() - t0) / 50
print(f"{sec_per_step:.2f} sec/step (micro-batch, no grad-accum)")

# estimate steps you can afford. e.g. plan 7 hours of training:
hours = 7
full_step_time = sec_per_step * GRAD_ACCUM_STEPS
print(f"~{int(hours*3600/full_step_time)} full optimizer steps in {hours}h "
      f"-> set MAX_STEPS near this in Cell 2")

/tmp/ipykernel_23/2885746828.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()   # required for fp16


0.58 sec/step (micro-batch, no grad-accum)
~10810 full optimizer steps in 7h -> set MAX_STEPS near this in Cell 2


## Cell 8

In [9]:
def train_one(use_rope, max_steps):
    tag = "rope" if use_rope else "baseline"
    ckpt_dir = f"/kaggle/working/ckpt_{tag}"
    os.makedirs(ckpt_dir, exist_ok=True)
    ckpt_path = f"{ckpt_dir}/last.pt"

    # fresh config + fresh model for this run
    cfg = dict(CONFIG); cfg["use_rope"] = use_rope
    model = GPT(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n===== Training {tag} | params {n_params/1e6:.1f}M | use_rope={use_rope} =====")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9,0.95), weight_decay=0.1)
    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return step / max(1, WARMUP_STEPS)
        prog = (step - WARMUP_STEPS) / max(1, max_steps - WARMUP_STEPS)
        return 0.5 * (1 + math.cos(math.pi * min(prog, 1.0)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.cuda.amp.GradScaler()

    start_step, losses = 0, []
    if os.path.exists(ckpt_path):                      # resume if the commit restarts
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optim"])
        scheduler.load_state_dict(ck["sched"]); start_step = ck["step"]; losses = ck.get("losses", [])
        print(f"resumed {tag} from step {start_step}")
        if start_step >= max_steps:
            print(f"{tag} already complete, skipping")
            return model                                # return finished model for eval

    model.train()
    data_iter = iter(train_loader)
    t0 = time.time()
    for step in range(start_step, max_steps):
        optimizer.zero_grad(set_to_none=True)
        accum = 0.0
        for _ in range(GRAD_ACCUM_STEPS):
            x, y = next(data_iter); x, y = x.to(device), y.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                _, loss = model(x, y); loss = loss / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward(); accum += loss.item()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        losses.append(accum)

        if step % 50 == 0:
            print(f"[{tag}] step {step} | loss {accum:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | {time.time()-t0:.1f}s")
            t0 = time.time()
        if step % 500 == 0 and step > 0:
            torch.save({"model":model.state_dict(),"optim":optimizer.state_dict(),
                        "sched":scheduler.state_dict(),"step":step,"losses":losses}, ckpt_path)

    torch.save({"model":model.state_dict(),"optim":optimizer.state_dict(),
                "sched":scheduler.state_dict(),"step":max_steps,"losses":losses}, ckpt_path)
    with open(f"{ckpt_dir}/losses.json","w") as f:
        json.dump(losses, f)
    print(f"{tag} DONE ({max_steps} steps)")
    return model

STEPS_EACH = 8000     # 2 x 8000 = 16k steps ~= 8.5h, safe within the 12h commit limit

model_baseline = train_one(use_rope=False, max_steps=STEPS_EACH)
model_rope     = train_one(use_rope=True,  max_steps=STEPS_EACH)


===== Training baseline | params 110.0M | use_rope=False =====


/tmp/ipykernel_23/1064074144.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[baseline] step 0 | loss 10.4775 | lr 1.00e-06 | 8.4s
[baseline] step 50 | loss 7.4140 | lr 5.10e-05 | 90.1s
[baseline] step 100 | loss 4.9068 | lr 1.01e-04 | 91.1s
[baseline] step 150 | loss 4.5047 | lr 1.51e-04 | 91.1s
[baseline] step 200 | loss 4.5038 | lr 2.01e-04 | 91.1s
[baseline] step 250 | loss 4.3681 | lr 2.51e-04 | 91.1s
[baseline] step 300 | loss 4.5747 | lr 3.00e-04 | 91.0s
[baseline] step 350 | loss 4.0385 | lr 3.00e-04 | 91.1s
[baseline] step 400 | loss 4.1813 | lr 3.00e-04 | 91.1s
[baseline] step 450 | loss 4.1847 | lr 3.00e-04 | 91.1s
[baseline] step 500 | loss 4.0658 | lr 2.99e-04 | 91.1s
[baseline] step 550 | loss 3.6369 | lr 2.99e-04 | 92.9s
[baseline] step 600 | loss 4.0433 | lr 2.99e-04 | 91.1s
[baseline] step 650 | loss 3.8540 | lr 2.98e-04 | 91.1s
[baseline] step 700 | loss 3.9851 | lr 2.98e-04 | 91.1s
[baseline] step 750 | loss 3.9998 | lr 2.97e-04 | 91.1s
[baseline] step 800 | loss 4.3875 | lr 2.97e-04 | 91.1s
[baseline] step 850 | loss 3.8202 | lr 2.96e-04 | 9

In [10]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

utilization.gpu [%], memory.used [MiB]
0 %, 11719 MiB
0 %, 3 MiB


## Cell 9

In [11]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=80, temperature=0.8, top_k=40):
    model.eval()
    x = torch.tensor([tok.encode(prompt).ids], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        x_cond = x[:, -CONFIG["block_size"]:]
        logits, _ = model(x_cond)
        logits = logits[:, -1, :] / temperature
        v, _ = torch.topk(logits, top_k)
        logits[logits < v[:, [-1]]] = -float("inf")
        probs = F.softmax(logits, dim=-1)
        x = torch.cat([x, torch.multinomial(probs, 1)], dim=1)
    model.train()
    return tok.decode(x[0].tolist())

prompts = ["The history of", "भारत एक", "महाराष्ट्र ही"]
for name, m in [("BASELINE", model_baseline), ("RoPE", model_rope)]:
    print(f"\n########## {name} ##########")
    for p in prompts:
        print(f"{p}  ->  {generate(m, p)}\n")


########## BASELINE ##########
The history of  ->  The history of a great place in the same time. This is also a long place to be the first time to live, but not the most important, it is the best choice for a single.
But what is a great place for the whole week?
My first day, I would love to be a great way to have the best price for both the best.
I do not love the work.

भारत एक  ->  भारत एक विषय से परिचित है।
देश की नीतियाँ जानिए और सिनेमा देश होने जा रहे हैं, उनकी बातों में एक बार फिर से सुनने और लोगों को जीवन

महाराष्ट्र ही  ->  महाराष्ट्र ही विधानसभा निवडणुकीची लाट असतानाही या प्रसंगातील मुख्यमंत्र्यांनी आपल्या प्रतिक्रियेतून ही सुटका केली आहे.
आं


########## RoPE ##########
The history of  ->  The history of the S.J. and the American History of America.
The State of America is a popular source of the World War II. The history, the United States has an important impact on the American Republic of India.
The United States is also a national-state-based approach of the United Nat

## Cell 10

In [12]:
@torch.no_grad()
def eval_perplexity(model, lang, n_batches=50):
    model.eval()
    buf, losses, count = [], [], 0
    for text in lang_stream(lang):
        buf.extend(tok.encode(text).ids + [EOT_ID])
        while len(buf) >= CONFIG["block_size"] + 1:
            chunk = buf[:CONFIG["block_size"]+1]; buf = buf[CONFIG["block_size"]:]
            x = torch.tensor([chunk[:-1]], device=device)
            y = torch.tensor([chunk[1:]],  device=device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                _, loss = model(x, y)
            losses.append(loss.item()); count += 1
            if count >= n_batches: break
        if count >= n_batches: break
    model.train()
    avg = sum(losses)/len(losses)
    return avg, math.exp(avg)

print(f"{'lang':<6}{'baseline loss':<16}{'baseline ppl':<16}{'rope loss':<16}{'rope ppl':<10}")
results = {}
for lang in LANGS:
    bl, bppl = eval_perplexity(model_baseline, lang)
    rl, rppl = eval_perplexity(model_rope, lang)
    results[lang] = dict(baseline_loss=bl, baseline_ppl=bppl, rope_loss=rl, rope_ppl=rppl)
    print(f"{lang:<6}{bl:<16.3f}{bppl:<16.1f}{rl:<16.3f}{rppl:<10.1f}")

with open("/kaggle/working/perplexity_results.json","w") as f:
    json.dump(results, f, indent=2)

lang  baseline loss   baseline ppl    rope loss       rope ppl  


Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

mr    1.879           6.5             1.835           6.3       
